In [4]:
import pandas as pd
import numpy as np
import random
import math
import os
import operator

# ====================================================================================================
# --- DYNAMIC WEIGHT ADJUSTMENT (DWA) PARAMETERS ---
# These parameters control how the system adapts W_L and W_A in real-time.
# ====================================================================================================

# DWA Goals
TARGET_REROUTE_RATE = 0.95 # Goal: 95% of primary failures should be successfully rerouted
TARGET_BACKUP_ALLOCATION_RATE = 0.75 # NEW: Goal: 75% of requests should have a pre-allocated backup
DYNAMIC_WEIGHT_LEARNING_RATE = 0.2 # How aggressively weights are adjusted (0.0 to 1.0)
MIN_WEIGHT = 0.1 # Minimum floor for W_LAT or W_AVL
MAX_WEIGHT = 0.9 # Maximum ceiling for W_LAT or W_AVL


# ====================================================================================================
# --- SIMULATION PARAMETERS (MODIFIED FOR SCARCITY AND STRESS) ---
# ====================================================================================================

EDGES_SELECT = 40 # Total number of edge nodes selected
USERS_SELECT = 600 # Total number of users selected for the simulation
SERVICES = 4 # Fixed to 3 services
BUDGET = 3500 # Total budget for service deployment

# --- Request & Failure Model (HIGH STRESS) ---
FIXED_REQUESTS_PER_SLOT = 600 # INCREASED load to stress capacity
FIXED_FAILURES_PER_SLOT = 5 # INCREASED failures (50% of edges fail)


LARGE_SLOT_LENGTH = 10 # Placement update every 10 short slots
SIM_SHORT_SLOTS = 120 # Total simulation duration for readable trace

# --- Latency model ---
SPEED_MS_PER_KM = 5.0

# --- Files ---
EDGE_CSV = "edgecbd.csv"
USER_CSV = "usercbd.csv"

# --- Service Criticality Dynamic Assignment ---
SERVICE_CRITICALITY = [0.9, 0.99, 0.999, 0.9999, 0.99999] # Increased to push W_A score higher

# --- History for demand prediction and failure tracking ---
HISTORY_LIMIT = 10

# =========================
# ATTRIBUTES
# =========================
# Assign the chosen service criticalities to the currently used SERVICES count
service_avail = SERVICE_CRITICALITY[:SERVICES]
while len(service_avail) < SERVICES:
    service_avail.append(0.9) # Default criticality if not enough defined


# Service attributes
SERVICE_COST = [random.randint(40, 120) for _ in range(SERVICES)]
SERVICE_LATENCY_LIMIT_MS = [random.randint(10, 1000) for _ in range(SERVICES)]
# Calculate a target latency for DWA, using half the average limit as a goal
TARGET_LATENCY = np.mean(SERVICE_LATENCY_LIMIT_MS) / 2    

# =========================
# DATA LOADING (CSV) + SUBSAMPLING (Dummy data if CSVs are missing)
# =========================
def load_points(path: str):
    '''Loads spatial points from CSV or creates dummy data if file is missing.'''
    try:
        df = pd.read_csv(path)
        lat_col, lon_col = None, None

        # 1. Search for Latitude column name
        for c in df.columns:
            if 'lat' in c.lower():
                lat_col = c
                break
        # 2. Search for Longitude column name
        for c in df.columns:
            if 'lon' in c.lower() or 'lng' in c.lower():
                lon_col = c
                break

        if lat_col and lon_col:
            # Use the dynamically detected column names
            pts = df[[lat_col, lon_col]].to_numpy(dtype=float)
        else:
            # Fallback to the first two numeric columns
            numeric = df.select_dtypes(include=[np.number]).columns
            if len(numeric) >= 2:
                pts = df[[numeric[0], numeric[1]]].to_numpy(dtype=float)
            else:
                raise ValueError(f"Cannot detect coordinates in {path}")
        return pts
    except FileNotFoundError:
        print(f"Creating dummy data for {path}...")
        if "edge" in path:
            # Create a small CSV for edge nodes
            pd.DataFrame({'lat': np.random.uniform(30, 40, 100), 'lon': np.random.uniform(-100, -90, 100)}).to_csv(path, index=False)
        else:
            # Create a larger CSV for user nodes
            pd.DataFrame({'lat': np.random.uniform(30, 40, 1000), 'lon': np.random.uniform(-100, -90, 1000)}).to_csv(path, index=False)
        return load_points(path)


# Ensure the dummy files exist if the real ones don't
if not os.path.exists(EDGE_CSV):
    load_points(EDGE_CSV)
if not os.path.exists(USER_CSV):
    load_points(USER_CSV)

edges_xy_all = load_points(EDGE_CSV)
users_xy_all = load_points(USER_CSV)

NUM_EDGES_TOTAL = len(edges_xy_all)
NUM_USERS_TOTAL = len(users_xy_all)

# Safely select subsets
edge_idx = random.sample(range(NUM_EDGES_TOTAL), EDGES_SELECT)
user_idx = random.sample(range(NUM_USERS_TOTAL), USERS_SELECT)

edges_xy = edges_xy_all[edge_idx]
users_xy = users_xy_all[user_idx]

NUM_EDGES = EDGES_SELECT
NUM_USERS = USERS_SELECT

edges = [{'id': i, 'lat': float(edges_xy[i, 0]), 'lon': float(edges_xy[i, 1])} for i in range(NUM_EDGES)]
users = [{'id': i, 'lat': float(users_xy[i, 0]), 'lon': float(users_xy[i, 1])} for i in range(NUM_USERS)]

# Edge attributes
coverage_radius_m = [random.randint(200, 1000) for _ in range(NUM_EDGES)]
avail_service_slots = [random.randint(2, SERVICES) for _ in range(NUM_EDGES)]
maxrequests = [random.randint(50, 150) for _ in range(NUM_EDGES)]    

cost = np.zeros((NUM_EDGES, SERVICES), dtype=int)
for e in range(NUM_EDGES):
    for s in range(SERVICES):
        cost[e, s] = SERVICE_COST[s]

# =========================
# GEO UTILS + LATENCY
# ====================================================================================================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dLat = math.radians(lat2 - lat1)
    dLon = math.radians(lon2 - lon1)
    a = math.sin(dLat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dLon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

km_dist = np.zeros((NUM_USERS, NUM_EDGES), dtype=float)
latency_ms = np.zeros_like(km_dist)
for u in range(NUM_USERS):
    for e in range(NUM_EDGES):
        dkm = haversine_km(users[u]['lat'], users[u]['lon'], edges[e]['lat'], edges[e]['lon'])
        km_dist[u, e] = dkm
        latency_ms[u, e] = dkm * SPEED_MS_PER_KM

# Pre-compute user covered edges, sorted by lowest latency
usercovered = []
for u in range(NUM_USERS):
    edgelists = {}
    edgelist = []
    for e in range(NUM_EDGES):
        # NOTE: Coverage check is purely geographical, not QoS.
        if(km_dist[u, e] * 1000 <= coverage_radius_m[e]):
            edgelists[e] = latency_ms[u, e]
    sort_listedges = sorted(edgelists.items(), key=operator.itemgetter(1), reverse=False)
    for key, value in sort_listedges:
        edgelist.append(key)
    usercovered.append(edgelist)

# =========================
# REDUNDANCY CHECK FOR PLACEMENT
# =========================
def find_qos_compliant_backup(primary_edge, service_id, temp_placement):
    """
    Checks if there is *any* other edge (not the primary) that:
    1. Has the service placed (or is available to be placed in temp_placement).
    2. Has at least one user that can reach it within the QoS latency limit.
    """
    s = service_id

    # Check for *any* other edge that can host this service and meet QoS
    for e_backup in range(NUM_EDGES):
        if e_backup != primary_edge:
            # 1. Check if placing here is possible (current placement + remaining slots)
            if (temp_placement[e_backup, s] == 1 or
                avail_service_slots[e_backup] > np.sum(temp_placement[e_backup, :])):

                # 2. Check if at least one user can reach this backup with QoS
                for u in range(NUM_USERS):
                    if latency_ms[u, e_backup] <= SERVICE_LATENCY_LIMIT_MS[s]:
                        return True # Found a viable, QoS-compliant backup location

    return False # No valid backup location found

# =========================
# DYNAMIC WEIGHT ADJUSTMENT (DWA) FUNCTION
# =========================
def dynamic_weight_adjustment(current_w_lat, current_w_avail, avg_latency, reroute_success_rate, avg_backup_allocation_rate):
    """
    Adjusts the weights W_LAT and W_AVL based on previous slot's performance.
    Now considers both Reroute Success Rate and Average Backup Allocation Rate for W_AVL.
    """
    # 1. Calculate Error Factors
    # High Latency -> High E_LAT (worse performance). Normalized by target.
    latency_error = (avg_latency - TARGET_LATENCY) / TARGET_LATENCY if TARGET_LATENCY > 0 else 0
    
    # Low Reroute Rate -> High E_RSR (worse performance)
    reroute_error = (TARGET_REROUTE_RATE - reroute_success_rate) 
    
    # Low Backup Allocation Rate -> High E_BAR (worse performance)
    backup_allocation_error = (TARGET_BACKUP_ALLOCATION_RATE - avg_backup_allocation_rate)
    
    # Combined Availability Error (E_AVL): Average of the two availability metrics errors.
    # Note: We are using equal weight (0.5) for both proactive (backup allocation) and reactive (reroute success) availability errors.
    combined_avail_error = reroute_error * 0.5 + backup_allocation_error * 0.5

    # 2. Calculate Adjustment Factor
    # Positive adjustment means latency is performing relatively worse, so W_LAT increases.
    # Negative adjustment means availability is performing relatively worse, so W_AVL increases (W_LAT decreases).
    adjustment = DYNAMIC_WEIGHT_LEARNING_RATE * (latency_error - combined_avail_error)

    # 3. Apply and Clip New Weights
    new_w_lat = current_w_lat + adjustment
    
    # Clip W_LAT to boundaries [MIN_WEIGHT, MAX_WEIGHT]
    new_w_lat = np.clip(new_w_lat, MIN_WEIGHT, MAX_WEIGHT)
    
    # Ensure W_LAT + W_AVL = 1
    new_w_avail = 1.0 - new_w_lat

    print(f"[DWA] Prev Latency Error: {latency_error:.3f}")
    print(f"[DWA] Reroute Error: {reroute_error:.3f}, Backup Allocation Error: {backup_allocation_error:.3f}")
    print(f"[DWA] Combined Avail Error: {combined_avail_error:.3f}. Adjustment: {adjustment:.3f}")
    print(f"[DWA] Weights updated: W_LAT={new_w_lat:.3f}, W_AVL={new_w_avail:.3f}")
    
    return new_w_lat, new_w_avail


# =========================
# ONLINE ADAPTIVE TWO-SCALE ALGORITHM
# =========================
def run_simulation(SIM_SHORT_SLOTS, LARGE_SLOT_LENGTH):
    
    # Initialize dynamic weights
    W_LATENCY = 0.5
    W_AVAIL = 0.5
    
    # Metrics
    total_latency_fulfilled = 0.0
    total_requests_fulfilled = 0
    total_requests_with_backup_assigned = 0
    total_routed_requests = 0
    service_counts_per_large_slot = []
    total_generated_requests_sim = 0
    total_primary_failures = 0
    total_proactive_backups_used = 0
    
    # History for DWA
    demand_history = np.zeros((HISTORY_LIMIT, NUM_EDGES, SERVICES))
    failure_history = np.zeros((HISTORY_LIMIT, NUM_EDGES))
    placement = np.zeros((NUM_EDGES, SERVICES), dtype=int)

    slot_trace = []


    # DWA Logic requires tracking metrics per large slot period
    large_slot_latency_sum = 0.0
    large_slot_requests_fulfilled = 0
    large_slot_primary_failures = 0
    large_slot_routed_requests = 0
    # NEW: Metrics for Backup Allocation Rate calculation in DWA
    large_slot_requests_with_backup_assigned = 0
    large_slot_generated_requests = 0


    def place_services(is_initial=False, t_slot=None, current_w_lat=0.5, current_w_avail=0.5):
        # Use nonlocal for variables modified inside the nested function
        nonlocal placement
        nonlocal service_counts_per_large_slot
        
        placement.fill(0)
        current_budget = BUDGET

        # EDGE RELIABILITY CALCULATION
        edge_failure_rate = np.mean(failure_history, axis=0)
        edge_reliability_score = 1 - edge_failure_rate
        
        slot_info = f"Initial Placement (t=0)" if is_initial else f"Large Slot Update (t={t_slot})"

        if is_initial:
            # Simple greedy placement for initialization (no scoring yet)
            all_placements = []
            for e in range(NUM_EDGES):
                for s in range(SERVICES):
                    all_placements.append((e, s))
            random.shuffle(all_placements)

            for e, s in all_placements:
                if (current_budget >= cost[e, s] and
                    avail_service_slots[e] > np.sum(placement[e, :]) and
                    placement[e, s] == 0):

                    temp_placement = np.copy(placement)
                    temp_placement[e, s] = 1

                    if find_qos_compliant_backup(e, s, temp_placement) or SERVICES == 1:
                        placement[e, s] = 1
                        current_budget -= cost[e, s]
                        
            print(f"\n[PLACEMENT UPDATE] --- {slot_info} ---")
            print(f"Remaining Budget: {current_budget}/{BUDGET}")
            print("-------------------------------------------------")
            return

        # Large-scale placement update based on DYNAMIC scores
        predicted_demand_raw = np.mean(demand_history, axis=0) if np.sum(demand_history) > 0 else np.zeros((NUM_EDGES, SERVICES))

        placement_scores = {}
        for e in range(NUM_EDGES):
            for s in range(SERVICES):
                demand_score = predicted_demand_raw[e, s]

                if demand_score > 0:
                    relevant_latencies = [latency_ms[u, e] for u in range(NUM_USERS) if latency_ms[u, e] <= SERVICE_LATENCY_LIMIT_MS[s]]
                    mean_latency_to_edge = np.mean(relevant_latencies) if relevant_latencies else 0.0

                    lat_limit = SERVICE_LATENCY_LIMIT_MS[s]
                    # Latency Score (0=bad, 1=good, closer to 1 is better)
                    latency_score = 1 - (mean_latency_to_edge / lat_limit) if lat_limit > 0 and mean_latency_to_edge > 0 else 0
                    
                    service_criticality = service_avail[s]
                    edge_reliability = edge_reliability_score[e]
                    availability_score = service_criticality * edge_reliability

                    # COMBINED SCORE: Uses DYNAMIC weights (W_LATENCY and W_AVAIL)
                    combined_score = demand_score * (current_w_lat * latency_score +
                                                     current_w_avail * availability_score)

                    # --- REDUNDANCY AWARENESS CHECK (Penalty for non-redundancy) ---
                    temp_placement = np.copy(placement)
                    # Check if placing at (e, s) is the only instance of service s
                    if np.sum(placement[:, s]) == 0 and avail_service_slots[e] > np.sum(temp_placement[e, :]):
                        temp_placement[e, s] = 1
                        if not find_qos_compliant_backup(e, s, temp_placement):
                            combined_score *= 0.1 # Severe penalty to discourage non-redundant initial placement

                    placement_scores[(e, s)] = combined_score

        sorted_scores = sorted(placement_scores.items(), key=lambda item: item[1], reverse=True)

        # Apply the greedy placement based on the new redundancy-aware scores
        for (e, s), score in sorted_scores:
            if (current_budget >= cost[e, s] and
                avail_service_slots[e] > np.sum(placement[e, :]) and
                placement[e, s] == 0): # Check that it hasn't been placed yet

                placement[e, s] = 1
                current_budget -= cost[e, s]

        # --- Print Placement Matrix ---
        print(f"\n[PLACEMENT UPDATE] --- {slot_info} ---")
        print(f"Weights used: W_LAT={current_w_lat:.3f}, W_AVL={current_w_avail:.3f}")
        print(f"Remaining Budget: {current_budget}/{BUDGET}")
        print("-------------------------------------------------")
        # --- End Print ---

    # Initial placement must happen before the loop starts
    place_services(is_initial=True, current_w_lat=W_LATENCY, current_w_avail=W_AVAIL)
    service_counts_per_large_slot.append(np.sum(placement))

    
    for t in range(SIM_SHORT_SLOTS):

        # --- Dynamic Weight Adjustment and Placement Update ---
        if t > 0 and t % LARGE_SLOT_LENGTH == 0:
            
            # 1. Calculate performance for the previous large slot period
            
            # Calculate Reroute Success Rate for DWA
            reroute_success_rate = large_slot_routed_requests / large_slot_primary_failures if large_slot_primary_failures > 0 else 1.0

            # Calculate Average Latency for DWA
            avg_latency = large_slot_latency_sum / large_slot_requests_fulfilled if large_slot_requests_fulfilled > 0 else TARGET_LATENCY
            
            # NEW: Calculate Backup Allocation Rate for DWA
            large_slot_backup_allocation_rate = large_slot_requests_with_backup_assigned / large_slot_generated_requests if large_slot_generated_requests > 0 else 0.0
            
            # 2. Dynamic Weight Adjustment (UPDATED CALL)
            W_LATENCY, W_AVAIL = dynamic_weight_adjustment(
                W_LATENCY, 
                W_AVAIL, 
                avg_latency, 
                reroute_success_rate,
                large_slot_backup_allocation_rate # NEW ARGUMENT
            )
            
            # 3. Trigger Placement Update with new weights
            place_services(t_slot=t, current_w_lat=W_LATENCY, current_w_avail=W_AVAIL)
            service_counts_per_large_slot.append(np.sum(placement))
            
            # 4. Reset Large Slot counters
            large_slot_latency_sum = 0.0
            large_slot_requests_fulfilled = 0
            large_slot_primary_failures = 0
            large_slot_routed_requests = 0
            # NEW RESETS
            large_slot_requests_with_backup_assigned = 0
            large_slot_generated_requests = 0


        # --- Faults ---
        num_failures = FIXED_FAILURES_PER_SLOT
        num_failures = min(num_failures, NUM_EDGES)
        if num_failures > NUM_EDGES:
            failed_edges = range(NUM_EDGES)
        else:
            failed_edges = random.sample(range(NUM_EDGES), num_failures)    

        # --- Generate requests ---
        num_requests_this_slot = FIXED_REQUESTS_PER_SLOT
        current_requests = [{'user_id': random.randint(0, NUM_USERS - 1), 'service_id': random.randint(0, SERVICES - 1)} for _ in range(num_requests_this_slot)]
        total_generated_requests_sim += len(current_requests)
        large_slot_generated_requests += len(current_requests) # NEW AGGREGATION

        active_placement = np.copy(placement)

        # Capacity pool for both Primary and Backup reservation
        priback_cap = maxrequests.copy()

        # --- PROACTIVE PRIMARY AND BACKUP ALLOCATION (LOCAL TRADE-OFF) ---
        proactive_assignments = []
        slot_requests_with_backup_assigned = 0

        for req in current_requests:
            u, s = req['user_id'], req['service_id']
            best_primary_score = -1
            best_primary_edge = -1

            # 1. Primary Allocation
            for e in range(NUM_EDGES):
                if active_placement[e, s] == 1 and priback_cap[e] > 0:
                    if latency_ms[u, e] <= SERVICE_LATENCY_LIMIT_MS[s]:
                        # Latency Score (0=bad, 1=good)
                        lat_score = 1 - (latency_ms[u, e] / SERVICE_LATENCY_LIMIT_MS[s])
                        avail_criticality_score = service_avail[s]

                        # Primary score uses CURRENT DYNAMIC weights
                        combined_score = W_LATENCY * lat_score + W_AVAIL * avail_criticality_score

                        if combined_score > best_primary_score:
                            best_primary_score = combined_score
                            best_primary_edge = e

            # 2. Backup Allocation
            best_backup_edge = -1
            if best_primary_edge != -1:
                best_backup_score = -1

                # Search for backup on other edges, prioritizing closer ones (usercovered)
                for e in usercovered[u]:
                    if e != best_primary_edge and active_placement[e, s] == 1 and priback_cap[e] > 0:
                        if latency_ms[u, e] <= SERVICE_LATENCY_LIMIT_MS[s]:

                            backup_lat_score = 1 - (latency_ms[u, e] / SERVICE_LATENCY_LIMIT_MS[s])
                            backup_crit_score = service_avail[s]

                            # Backup Score uses CURRENT DYNAMIC weights.
                            combined_score = (W_AVAIL * backup_crit_score) + (W_LATENCY * backup_lat_score)

                            if combined_score > best_backup_score:
                                best_backup_score = combined_score
                                best_backup_edge = e

            if best_primary_edge != -1:
                # Primary successfully allocated (consume capacity)
                priback_cap[best_primary_edge] -= 1

                # Assign backup capacity and increment metric if a backup was found (consume capacity)
                if best_backup_edge != -1:
                    priback_cap[best_backup_edge] -= 1
                    slot_requests_with_backup_assigned += 1

                proactive_assignments.append({
                    'user_id': u, 'service_id': s, 'primary_edge': best_primary_edge,
                    'backup_edge': best_backup_edge, 'status': 'primary'
                })

        total_requests_with_backup_assigned += slot_requests_with_backup_assigned
        large_slot_requests_with_backup_assigned += slot_requests_with_backup_assigned # NEW AGGREGATION

        # --- 3. REACTIVE FAULT HANDLING (Re-routing) ---
        slot_proactive_backups_used = 0
        slot_total_routed_requests = 0
        slot_primary_failures = 0
        
        # Track which backup servers actually handled the load in this slot
        backup_consumption = {e: 0 for e in range(NUM_EDGES)}

        for assign in proactive_assignments:
            is_primary_failed = assign['primary_edge'] in failed_edges
            is_backup_valid = assign['backup_edge'] != -1
            is_backup_failed = is_backup_valid and assign['backup_edge'] in failed_edges

            u, s = assign['user_id'], assign['service_id']

            if is_primary_failed:
                slot_primary_failures += 1
                
                # Proactive Reroute Attempt
                if is_backup_valid and not is_backup_failed:
                    backup_edge_id = assign['backup_edge']
                    
                    if backup_consumption[backup_edge_id] < maxrequests[backup_edge_id] and \
                       latency_ms[u, backup_edge_id] <= SERVICE_LATENCY_LIMIT_MS[s]:
                        
                        # Reroute successful to proactive backup
                        assign['status'] = 'backup'
                        assign['latency'] = latency_ms[u, backup_edge_id]
                        slot_proactive_backups_used += 1
                        slot_total_routed_requests += 1
                        backup_consumption[backup_edge_id] += 1
                    else:
                        # Backup failed due to Capacity Contention or Latency
                        assign['status'] = 'failed_proactive_contention'    
                else:
                    # Reactive Reroute Attempt (Only if proactive failed/didn't exist)
                    best_reactive_score = -1
                    best_reactive_edge = -1

                    for e in range(NUM_EDGES):
                        if e not in failed_edges and active_placement[e, s] == 1:
                            if backup_consumption[e] < maxrequests[e]: # Check available real capacity
                                if latency_ms[u, e] <= SERVICE_LATENCY_LIMIT_MS[s]:
                                    lat_score = 1 - (latency_ms[u, e] / SERVICE_LATENCY_LIMIT_MS[s])
                                    avail_criticality_score = service_avail[s]
                                    
                                    # Reactive reroute also uses CURRENT DYNAMIC weights
                                    combined_score = W_LATENCY * lat_score + W_AVAIL * avail_criticality_score

                                    if combined_score > best_reactive_score:
                                        best_reactive_score = combined_score
                                        best_reactive_edge = e

                    if best_reactive_edge != -1:
                        # Reroute successful to reactive edge
                        assign['status'] = 'reactive_reroute'
                        assign['latency'] = latency_ms[u, best_reactive_edge]
                        slot_total_routed_requests += 1
                        backup_consumption[best_reactive_edge] += 1
                    else:
                        assign['status'] = 'failed'
            else:
                # Primary successful
                assign['status'] = 'primary'
                assign['latency'] = latency_ms[u, assign['primary_edge']]

        total_proactive_backups_used += slot_proactive_backups_used
        total_routed_requests += slot_total_routed_requests
        total_primary_failures += slot_primary_failures

        slot_latency_fulfilled = 0.0
        slot_requests_fulfilled = 0
        slot_demand = np.zeros((NUM_EDGES, SERVICES))

        for assign in proactive_assignments:
            if assign['status'] in ['primary', 'backup', 'reactive_reroute']:
                slot_requests_fulfilled += 1
                slot_latency_fulfilled += assign['latency']
                
                # Demand only tracks successful primary routing for the next large slot placement
                if assign['status'] == 'primary':    
                    if assign['primary_edge'] != -1:
                        slot_demand[assign['primary_edge'], assign['service_id']] += 1

        total_requests_fulfilled += slot_requests_fulfilled
        total_latency_fulfilled += slot_latency_fulfilled
        
        # Aggregate large slot metrics
        large_slot_latency_sum += slot_latency_fulfilled
        large_slot_requests_fulfilled += slot_requests_fulfilled
        large_slot_primary_failures += slot_primary_failures
        large_slot_routed_requests += slot_total_routed_requests

        # Calculate per-slot metrics for the trace
        slot_avg_latency = slot_latency_fulfilled / slot_requests_fulfilled if slot_requests_fulfilled > 0 else 0.0
        
        # --- Print Slot Metrics ---
        failure_list_str = [str(e) for e in failed_edges]    
        print(f"--- Slot {t} Results (W_L={W_LATENCY:.3f}, W_A={W_AVAIL:.3f}) ---")
        print(f"  Number of failed edges: {num_failures} (Edges: {', '.join(failure_list_str[:5])}... and {len(failure_list_str) - 5} more)")
        print(f"  Generated requests: {num_requests_this_slot}")
        print(f"  Fulfilled Requests: {slot_requests_fulfilled}/{num_requests_this_slot}")
        print(f"  Primary Failures: {slot_primary_failures}")
        print(f"  Requests rerouted (proactive/reactive): {slot_total_routed_requests}")
        print(f"  Proactive Backups Used: {slot_proactive_backups_used}")
        print(f"  Avg Latency: {slot_avg_latency:.2f} ms")
        print(f"---------------------------------------------")
        # --- End Print ---

        # Update demand history
        demand_history = np.roll(demand_history, -1, axis=0)
        demand_history[-1, :, :] = slot_demand

        # Update failure history
        failure_mask = np.zeros(NUM_EDGES)
        failure_mask[failed_edges] = 1
        failure_history = np.roll(failure_history, -1, axis=0)
        failure_history[-1, :] = failure_mask

        slot_trace.append({
            'slot': t,
            'requests_fulfilled': slot_requests_fulfilled,
            'avg_latency_ms': slot_avg_latency,
            'backup_allocation_rate': slot_requests_with_backup_assigned / num_requests_this_slot if num_requests_this_slot > 0 else 0.0
        })

    # --- Final Metric Calculations ---
    total_generated_requests = total_generated_requests_sim
    avg_latency = total_latency_fulfilled / total_requests_fulfilled if total_requests_fulfilled > 0 else 0.0
    avg_backup_allocation_rate = total_requests_with_backup_assigned / total_generated_requests if total_generated_requests > 0 else 0.0

    # Reroute Success Rate must be against the number of *primary failures* that needed a reroute
    reroute_success_rate = total_routed_requests / total_primary_failures if total_primary_failures > 0 else 0.0

    avg_coverage = total_requests_fulfilled / total_generated_requests if total_generated_requests > 0 else 0.0
    avg_deployments = np.mean(service_counts_per_large_slot) if service_counts_per_large_slot else 0.0
    avg_backup_usage_per_slot = total_proactive_backups_used / SIM_SHORT_SLOTS
    avg_failures = FIXED_FAILURES_PER_SLOT    

    return avg_latency, avg_backup_allocation_rate, reroute_success_rate, avg_coverage, avg_deployments, avg_backup_usage_per_slot, avg_failures, slot_trace

if __name__ == "__main__":
    
    # Run the simulation once, DWA handles the weight adjustment
    (
        avg_latency,    
        avg_backup_allocation_rate,    
        reroute_success_rate,    
        avg_coverage,    
        avg_deployments,    
        avg_backup_usage_per_slot,    
        avg_failures,    
        slot_trace_data
    ) = run_simulation(SIM_SHORT_SLOTS, LARGE_SLOT_LENGTH)
    

    print("\n=========================")
    print("      SIMULATION RESULTS (Dynamic Weight Adjustment)")
    print("=========================")
    print(f"1. Average User Latency Coverage Rate (Fulfilled / Generated): {avg_coverage:.2%}")
    print(f"2. Average Backup Allocation Rate (Requests Allocated Backup): {avg_backup_allocation_rate:.2%}")
    print(f"3. Reroute Success Rate (Rerouted / Primary Failed): {reroute_success_rate:.2%}")
    print("-------------------------")
    print(f"Target Latency for DWA: {TARGET_LATENCY:.2f} ms")
    print(f"Target Reroute Rate for DWA: {TARGET_REROUTE_RATE:.2%}")
    print(f"Target Backup Allocation Rate for DWA: {TARGET_BACKUP_ALLOCATION_RATE:.2%}")
    print(f"Average latency for fulfilled requests: {avg_latency:.2f} ms")
    print(f"Average service deployments per large slot: {avg_deployments:.2f}")
    print(f"Average backup servers used for failover per short slot: {avg_backup_usage_per_slot:.2f}")
    print("================================================================================================")



[PLACEMENT UPDATE] --- Initial Placement (t=0) ---
Remaining Budget: 42/3500
-------------------------------------------------
--- Slot 0 Results (W_L=0.500, W_A=0.500) ---
  Number of failed edges: 5 (Edges: 10, 20, 14, 2, 1... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 72
  Requests rerouted (proactive/reactive): 72
  Proactive Backups Used: 62
  Avg Latency: 1.14 ms
---------------------------------------------
--- Slot 1 Results (W_L=0.500, W_A=0.500) ---
  Number of failed edges: 5 (Edges: 39, 10, 11, 33, 25... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 85
  Requests rerouted (proactive/reactive): 85
  Proactive Backups Used: 81
  Avg Latency: 1.14 ms
---------------------------------------------
--- Slot 2 Results (W_L=0.500, W_A=0.500) ---
  Number of failed edges: 5 (Edges: 29, 19, 3, 23, 10... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 99
  Re

--- Slot 26 Results (W_L=0.139, W_A=0.861) ---
  Number of failed edges: 5 (Edges: 37, 21, 10, 0, 4... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 153
  Requests rerouted (proactive/reactive): 153
  Proactive Backups Used: 120
  Avg Latency: 1.32 ms
---------------------------------------------
--- Slot 27 Results (W_L=0.139, W_A=0.861) ---
  Number of failed edges: 5 (Edges: 9, 32, 19, 27, 23... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 92
  Requests rerouted (proactive/reactive): 92
  Proactive Backups Used: 71
  Avg Latency: 1.20 ms
---------------------------------------------
--- Slot 28 Results (W_L=0.139, W_A=0.861) ---
  Number of failed edges: 5 (Edges: 5, 20, 26, 8, 0... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 91
  Requests rerouted (proactive/reactive): 91
  Proactive Backups Used: 61
  Avg Latency: 1.25 ms
--------------------------------

--- Slot 54 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 19, 38, 21, 15, 18... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 134
  Requests rerouted (proactive/reactive): 134
  Proactive Backups Used: 106
  Avg Latency: 1.28 ms
---------------------------------------------
--- Slot 55 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 9, 2, 8, 11, 26... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 71
  Requests rerouted (proactive/reactive): 71
  Proactive Backups Used: 51
  Avg Latency: 1.16 ms
---------------------------------------------
--- Slot 56 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 1, 37, 39, 2, 11... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 63
  Requests rerouted (proactive/reactive): 63
  Proactive Backups Used: 55
  Avg Latency: 1.13 ms
-------------------------------

--- Slot 80 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 20, 5, 25, 30, 6... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 41
  Requests rerouted (proactive/reactive): 41
  Proactive Backups Used: 41
  Avg Latency: 1.14 ms
---------------------------------------------
--- Slot 81 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 6, 2, 33, 11, 1... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 45
  Requests rerouted (proactive/reactive): 45
  Proactive Backups Used: 40
  Avg Latency: 1.15 ms
---------------------------------------------
--- Slot 82 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 3, 23, 16, 36, 29... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 44
  Requests rerouted (proactive/reactive): 44
  Proactive Backups Used: 44
  Avg Latency: 1.15 ms
-----------------------------------

--- Slot 104 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 18, 21, 0, 19, 25... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 149
  Requests rerouted (proactive/reactive): 149
  Proactive Backups Used: 89
  Avg Latency: 1.39 ms
---------------------------------------------
--- Slot 105 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 21, 24, 5, 36, 26... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 100
  Requests rerouted (proactive/reactive): 100
  Proactive Backups Used: 69
  Avg Latency: 1.18 ms
---------------------------------------------
--- Slot 106 Results (W_L=0.100, W_A=0.900) ---
  Number of failed edges: 5 (Edges: 21, 0, 4, 14, 26... and 0 more)
  Generated requests: 600
  Fulfilled Requests: 600/600
  Primary Failures: 148
  Requests rerouted (proactive/reactive): 148
  Proactive Backups Used: 108
  Avg Latency: 1.31 ms
-----------------------